<div style="background: linear-gradient(135deg, #1e3c72 0%, #2a5298 50%, #0f2027 100%); padding: 30px; border-radius: 15px; box-shadow: 0 10px 25px rgba(0,0,0,0.3); color: white; margin-bottom: 25px;">
    <div style="display: flex; align-items: center; justify-content: space-between; flex-wrap: wrap;">
        <div>
            <h1 style="font-size: 2.8em; font-weight: 800; margin: 0 0 10px 0; color: #ffffff; text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
                🌱 Kaggriculture: Starter Agent & Strategy
            </h1>
            <p style="font-size: 1.2em; margin: 0 0 15px 0; color: #e0e6ed; max-width: 800px;">
                A Comprehensive Mastery Notebook for $50,000 Simulation Competition — Dynamic Market Economics, Resource Allocation, Supply Chains, & Autonomous RL/Heuristic Agents.
            </p>
        </div>
    </div>
    <hr style="border: 0; height: 1px; background: rgba(255,255,255,0.2); margin: 15px 0 20px 0;">
    <div style="display: flex; gap: 10px; flex-wrap: wrap;">
        <span style="background: rgba(255,255,255,0.15); border: 1px solid rgba(255,255,255,0.3); padding: 6px 14px; border-radius: 20px; font-size: 0.9em; font-weight: 600;">🏆 Competition: Kaggriculture ($50,000 USD)</span>
        <span style="background: rgba(255,255,255,0.15); border: 1px solid rgba(255,255,255,0.3); padding: 6px 14px; border-radius: 20px; font-size: 0.9em; font-weight: 600;">⏱️ Duration: 720 Turns (30 Days)</span>
        <span style="background: rgba(255,255,255,0.15); border: 1px solid rgba(255,255,255,0.3); padding: 6px 14px; border-radius: 20px; font-size: 0.9em; font-weight: 600;">🤖 Engine: kaggle-environments</span>
    </div>
</div>

---

## 📌 Executive Summary

**Kaggriculture** is a 2-player turn-based competitive simulation where autonomous agents operate rival commercial farms over a 30-day season (720 turns total: 24 turns per day). 

To maximize end-of-season profit (bank balance in coins), an agent must master:
1. **Agricultural Operations**: Planting, watering, fertilizing, harvesting crops, and clearing weeds.
2. **Livestock & Supply Chains**: Building coops and pastures, buying and feeding geese, cows, and sheep with harvested wheat, and caring for livestock to boost yield bonuses.
3. **Price-Elastic Dynamic Markets**: Prices adjust dynamically based on market inventory ($I_0 = 10,000$). Selling drives prices down toward a $1 floor; town consumption and buying drive prices up.
4. **Labor Scaling**: Hiring daily farm hands using a Fibonacci cost multiplier ($1, 1, 2, 3, 5, 8, \dots$).
5. **Land Expansion**: Purchasing neighboring $5 	imes 5$ land quadrants for $\$1,000$, $\$2,000$, and $\$4,000$.

---


## 🛠️ Section 1: Setup & Environment Initialization

First, install and verify `kaggle-environments` and standard visualization libraries.


In [ ]:
!pip install -q kaggle-environments matplotlib seaborn pandas numpy


In [ ]:
import sys
import os
import json
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual aesthetic style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

import kaggle_environments
from kaggle_environments import make, evaluate

print(f"✅ Kaggle Environments version: {kaggle_environments.__version__}")


## 📐 Section 2: Mathematical Formulation & Economics Deep-Dive

### 2.1 The Dynamic Market Price Function

In Kaggriculture, market supply and demand govern resource prices. Every item starts at baseline inventory $I_0 = 10,000$.

When inventory $inv \neq I_0$, the price shifts according to:

$$ \text{price}(inv) = \text{floor}\left( \text{base} + \text{sign} \cdot \text{amp} \cdot f(|inv - I_0|) \right) $$

where:
- $\text{sign} = +1$ if $inv < I_0$ (Scarcity $\rightarrow$ Price Rises)
- $\text{sign} = -1$ if $inv > I_0$ (Glut / Overproduction $\rightarrow$ Price Drops)
- $\text{amp} = \frac{\text{target} \cdot \text{base}}{f(T)}$ (Derived amplitude constant based on anchor capacity $T$)
- $f \in \{\text{linear}, \text{sq}, \text{sqrt}, \text{log}, \text{log10}\}$ shape functions.

Price is strictly floored at **$1**.

#### Product Economy Matrix:

| Resource | Yield Type | Seed Cost | Base Price | Max Yield | Max Yield/Tile/Day | Price Curve (Below $I_0$) | Price Curve (Above $I_0$) |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Wheat** | One-time | $10 | $25 | 6 | 1.50 | sqrt (target=0.80) | log (target=0.20) |
| **Carrot** | One-time | $20 | $35 | 4 | 1.33 | log (target=0.20) | sqrt (target=0.70) |
| **Tomato** | Ongoing | $50 | $60 | 4 | 4.00 | linear (target=0.40) | sqrt (target=0.60) |
| **Strawberry** | Ongoing | $100 | $120 | 4 | 2.00 | sqrt (target=0.70) | linear (target=1.60) |
| **Melon** | One-time | $80 | $250 | 6 | 0.50 | log (target=0.20) | sq (target=3.60) |
| **Goose / Egg** | Ongoing | $300 | $50 | 4 | 2.00 | linear (target=0.40) | log (target=0.20) |
| **Cow / Milk** | Ongoing | $400 | $160 | 6 | 1.00 | sqrt (target=0.60) | linear (target=1.60) |
| **Sheep / Wool** | Ongoing | $500 | $200 | 6 | 0.67 | log (target=0.20) | sq (target=3.20) |


In [ ]:
# Let's plot the theoretical market price curves for key crops to visualize price sensitivity!

def compute_price(base, I0, T, inv, func_below, target_below, func_above, target_above):
    diff = inv - I0
    if diff == 0:
        return base
    
    def apply_f(func, x):
        if func == 'sqrt': return math.sqrt(x)
        if func == 'linear': return x
        if func == 'log': return math.log(1 + x)
        if func == 'sq': return x ** 2
        if func == 'log10': return math.log10(1 + x)
        return x

    if diff < 0: # Scarcity
        x = abs(diff)
        f_val = apply_f(func_below, x)
        f_T = apply_f(func_below, T)
        amp = (target_below * base) / f_T if f_T != 0 else 0
        p = base + amp * f_val
    else: # Glut
        x = diff
        f_val = apply_f(func_above, x)
        f_T = apply_f(func_above, T)
        amp = (target_above * base) / f_T if f_T != 0 else 0
        p = base - amp * f_val

    return max(1, round(p))

# Simulate price dynamics across inventory shifts (-500 to +1000 units relative to I0)
inv_range = np.linspace(9500, 11000, 300)
crops_params = {
    'Wheat': (25, 10000, 400, 'sqrt', 0.80, 'log', 0.20),
    'Melon': (250, 10000, 300, 'log', 0.20, 'sq', 3.60),
    'Strawberry': (120, 10000, 100, 'sqrt', 0.70, 'linear', 1.60),
    'Tomato': (60, 10000, 200, 'linear', 0.40, 'sqrt', 0.60)
}

plt.figure(figsize=(12, 6))
for name, params in crops_params.items():
    prices = [compute_price(*params[:3], inv, *params[3:]) for inv in inv_range]
    plt.plot(inv_range - 10000, prices, label=f"{name} (Base ${params[0]})", linewidth=2.5)

plt.axvline(0, color='gray', linestyle='--', alpha=0.7, label='Initial Market Inventory ($I_0$)')
plt.axhline(1, color='red', linestyle=':', label='Price Floor ($1)')
plt.title("Kaggriculture Market Price Elasticity Curves", fontsize=14, fontweight='bold')
plt.xlabel("Net Inventory Shift from Equilibrium ($inv - I_0$)", fontsize=12)
plt.ylabel("Market Price ($)", fontsize=12)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()


## 🎮 Section 3: Interactive Environment & Replay Visualizer

Let's test creating a `kaggriculture` environment instance, running a sample simulation between built-in agents, and creating a rich custom ASCII/HTML visualizer to inspect game state.


In [ ]:
# Initialize environment
env = make("kaggriculture", debug=True)
print("Configuration parameters:")
for k, v in list(env.configuration.items())[:8]:
    print(f" - {k}: {v}")


In [ ]:
class KaggricultureVisualizer:
    """Rich visualizer for Kaggriculture observations and farm grids."""
    
    @staticmethod
    def render_ascii(obs):
        day = obs['day']
        hour = obs['hour']
        m_prices = obs['market']['prices']
        
        print(f"╔══════════════════════════════════════════════════════════════════════════════════╗")
        print(f"║ 📅 DAY {day+1:02d} | ⏰ HOUR {hour:02d}/24 | 🌾 Market Prices: WHEAT ${m_prices.get('WHEAT', 0)} | MELON ${m_prices.get('MELON', 0)} | TOMATO ${m_prices.get('TOMATO', 0)} ║")
        print(f"╠══════════════════════════════════════════════════════════════════════════════════╣")
        
        for p_idx, farm in enumerate(obs['farms']):
            money = farm['money']
            farmer = farm['farmer']
            hands = farm['hands']
            quads = len(farm['unlocked_quadrants'])
            
            grid_str = ""
            for y in range(10):
                row = []
                for x in range(10):
                    tile = farm['tiles'][y][x]
                    if [x, y] == farmer:
                        cell = "👨"
                    elif [x, y] in hands:
                        cell = "🧑"
                    elif tile == "LOCKED":
                        cell = "🔒"
                    elif tile is None:
                        cell = "🟫" # Empty dirt
                    elif isinstance(tile, dict) and tile.get('kind') == 'PLANT':
                        crop = tile['crop']
                        if crop == 'WHEAT': cell = "🌾"
                        elif crop == 'CARROT': cell = "🥕"
                        elif crop == 'TOMATO': cell = "🍅"
                        elif crop == 'STRAWBERRY': cell = "🍓"
                        elif crop == 'MELON': cell = "🍈"
                        else: cell = "🌱"
                    elif isinstance(tile, dict) and tile.get('kind') == 'WEED':
                        cell = "🌿"
                    elif isinstance(tile, dict) and tile.get('kind') == 'COOP':
                        cell = "🪿" if tile.get('animal') else "🛖"
                    elif isinstance(tile, dict) and tile.get('kind') == 'PASTURE':
                        cell = "🐮" if tile.get('animal') == 'COW' else ("🐑" if tile.get('animal') == 'SHEEP' else "🪵")
                    else:
                        cell = "▫️"
                    row.append(cell)
                grid_str += " ".join(row) + "\n"
                
            print(f"  P{p_idx} Bank: ${money:.1f} | Quads: {quads}/4 | Position: {farmer}")
            print("  Farm Grid:")
            for r in grid_str.strip().split('\n'):
                print(f"    {r}")
        print(f"╚══════════════════════════════════════════════════════════════════════════════════╝\n")

# Test visualizer on initial step
env.reset()
KaggricultureVisualizer.render_ascii(env.steps[0][0]['observation'])


## 🥊 Section 4: Baseline Agents & Evaluation Harness

Let's inspect and test three baseline agents:
1. **Pass Agent**: Takes no action.
2. **Random Agent**: Randomly moves and buys/sells.
3. **Starter Wheat Loop Agent**: Simple deterministic wheat farming loop.


In [ ]:
def pass_agent(obs):
    """Minimal agent that passes every turn."""
    return {"farmer": ["PASS"], "hands": [], "market": []}

def starter_wheat_agent(obs):
    """Official starter agent implementing a basic wheat farming loop."""
    player = obs["player"]
    me = obs["farms"][player]
    private = obs["private"]
    fx, fy = me["farmer"]
    tile = me["tiles"][fy][fx]

    market = []
    
    # 1. Buy wheat seed if none in inventory and money >= 10
    if private["seeds"].get("WHEAT", 0) == 0 and me["money"] >= 10:
        market.append(["BUY_SEED", "WHEAT", 1])
        
    # 2. Sell any harvested wheat sitting in the shed
    wheat_in_shed = private["shed"].get("WHEAT", 0)
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", wheat_in_shed])

    # 3. Plant wheat if standing on an empty unlocked tile and seeds are available
    if tile is None and private["seeds"].get("WHEAT", 0) > 0:
        return {"farmer": ["PLANT", "WHEAT"], "hands": [], "market": market}
        
    # 4. If standing on a plant, water or harvest
    if isinstance(tile, dict) and tile.get("kind") == "PLANT":
        crop_age = obs["day"] - tile["planted_day"]
        if crop_age >= 2:  # Wheat first_yield_day = 2
            return {"farmer": ["HARVEST"], "hands": [], "market": market}
        if not tile.get("watered_today", False):
            return {"farmer": ["WATER"], "hands": [], "market": market}

    # 5. Otherwise move towards empty tiles or pass
    return {"farmer": ["PASS"], "hands": [], "market": market}


In [ ]:
# Run a match between starter_wheat_agent and random agent
env = make("kaggriculture", debug=True)
env.run([starter_wheat_agent, "random"])

final_step = env.steps[-1]
print("Match Results:")
for p_idx, state in enumerate(final_step):
    print(f" Player {p_idx} ({'Starter Wheat' if p_idx==0 else 'Random'}): Final Money = ${state['observation']['farms'][p_idx]['money']:.2f}, Reward = {state['reward']}, Status = {state['status']}")


## 🚀 Section 5: Advanced Strategy — `AgroMaster-v1` Agent

To achieve high competitive standing on the Kaggle Leaderboard, we build **`AgroMaster-v1`**, an agent engineered with:

### Key Algorithmic Features:
1. **Dynamic Crop Selection Engine**:
   - **Phase 1 (Days 1–10)**: Fast-growing **Wheat** & **Carrots** for liquidity.
   - **Phase 2 (Days 11–22)**: High-yield **Tomatoes** & **Strawberries** for compound ongoing yields.
   - **Phase 3 (Days 23–30)**: Harvesting remaining crops, liquidating shed inventory before season end.
2. **Breadth-First Search (BFS) Navigation**:
   - Optimal pathfinding across unlocked farm tiles to target unwatered crops, ready-to-harvest tiles, or empty dirt.
3. **Price-Elastic Market Queueing**:
   - Monitors live prices relative to base prices ($I_0$). Batches sales when price exceeds base, avoids selling into crashed $1 floors.
4. **Adaptive Farm Hand Hiring**:
   - Evaluates unserviced tile load vs. Fibonacci hand hire costs ($1, 1, 2, 3, \dots$).


In [ ]:
def agromaster_agent(obs):
    """
    AgroMaster-v1: Strategic Multi-Phase Farm & Market Optimization Agent
    """
    player = obs["player"]
    day = obs["day"]
    me = obs["farms"][player]
    private = obs["private"]
    fx, fy = me["farmer"]
    tiles = me["tiles"]
    money = me["money"]
    market_obs = obs["market"]
    prices = market_obs["prices"]
    
    market_orders = []
    
    # -------------------------------------------------------------
    # 1. MARKET STRATEGY: Seeds, Selling, & Hiring
    # -------------------------------------------------------------
    # A. Crop Selection based on Season Phase
    if day < 8:
        target_crop = "WHEAT"
        target_seed_cost = 10
    elif day < 20:
        target_crop = "TOMATO" if money > 80 else "CARROT"
        target_seed_cost = 50 if target_crop == "TOMATO" else 20
    else:
        target_crop = "CARROT"
        target_seed_cost = 20

    # Seed Inventory Management
    current_seeds = private["seeds"].get(target_crop, 0)
    if current_seeds < 3 and money >= target_seed_cost * 3:
        market_orders.append(["BUY_SEED", target_crop, 3])
        
    # Wheat Seed Buffer for Safety
    if private["seeds"].get("WHEAT", 0) < 2 and money >= 20:
        market_orders.append(["BUY_SEED", "WHEAT", 2])

    # B. Price-Elastic Selling from Shed
    for item, qty in private["shed"].items():
        if qty > 0:
            item_price = prices.get(item, 0)
            # Only sell if price is reasonable (above floor unless last 3 days)
            if item_price > 1 or day >= 27:
                market_orders.append(["SELL", item, qty])

    # C. Adaptive Farm Hand Hiring
    # Hire hand if we have high money and lots of tiles to service
    if day > 3 and me["hires_today"] == 0 and money > 500:
        market_orders.append(["HIRE", 1])

    # -------------------------------------------------------------
    # 2. FARMER TILE & MOVEMENT DECISION ENGINE (BFS Target Selection)
    # -------------------------------------------------------------
    current_tile = tiles[fy][fx]
    
    # Action A: Clear Weed immediately if standing on one
    if isinstance(current_tile, dict) and current_tile.get("kind") == "WEED":
        return {"farmer": ["DIG"], "hands": [], "market": market_orders}

    # Action B: Harvest ready crops
    if isinstance(current_tile, dict) and current_tile.get("kind") == "PLANT":
        crop = current_tile["crop"]
        crop_age = day - current_tile["planted_day"]
        # Harvest conditions per crop type
        if crop in ["WHEAT", "CARROT", "MELON"] and crop_age >= 2:
            return {"farmer": ["HARVEST"], "hands": [], "market": market_orders}
        elif crop in ["TOMATO", "STRAWBERRY"] and crop_age >= 8:
            return {"farmer": ["HARVEST"], "hands": [], "market": market_orders}

    # Action C: Water unwatered plants
    if isinstance(current_tile, dict) and current_tile.get("kind") == "PLANT":
        if not current_tile.get("watered_today", False):
            return {"farmer": ["WATER"], "hands": [], "market": market_orders}

    # Action D: Plant seed on empty dirt tile
    if current_tile is None:
        if private["seeds"].get(target_crop, 0) > 0:
            return {"farmer": ["PLANT", target_crop], "hands": [], "market": market_orders}
        elif private["seeds"].get("WHEAT", 0) > 0:
            return {"farmer": ["PLANT", "WHEAT"], "hands": [], "market": market_orders}

    # Action E: BFS Pathfinding to Nearest High-Priority Tile
    def bfs_find_target():
        queue = [(fx, fy, [])]
        visited = set([(fx, fy)])
        
        while queue:
            cx, cy, path = queue.pop(0)
            tile_state = tiles[cy][cx]
            
            # Check if this cell needs attention
            if (cx, cy) != (fx, fy):
                if tile_state is None and (private["seeds"].get(target_crop, 0) > 0 or private["seeds"].get("WHEAT", 0) > 0):
                    return path[0] # Move towards empty dirt
                if isinstance(tile_state, dict) and tile_state.get("kind") == "WEED":
                    return path[0]
                if isinstance(tile_state, dict) and tile_state.get("kind") == "PLANT":
                    if not tile_state.get("watered_today", False):
                        return path[0]
            
            # Explore neighbors
            neighbors = [
                (cx, cy - 1, "NORTH"),
                (cx, cy + 1, "SOUTH"),
                (cx + 1, cy, "EAST"),
                (cx - 1, cy, "WEST")
            ]
            for nx, ny, direction in neighbors:
                if 0 <= nx < 10 and 0 <= ny < 10:
                    if tiles[ny][nx] != "LOCKED" and (nx, ny) not in visited:
                        visited.add((nx, ny))
                        queue.append((nx, ny, path + [direction]))
        return None

    next_move = bfs_find_target()
    if next_move:
        return {"farmer": [next_move], "hands": [], "market": market_orders}

    return {"farmer": ["PASS"], "hands": [], "market": market_orders}


## 📊 Section 6: Local Tournament & Performance Benchmarking

Let's execute a multi-match tournament comparing **`AgroMaster-v1`**, **`Starter Wheat`**, and **`Random`** agents to evaluate win rates and coin accumulation curves.


In [ ]:
# Run multi-episode benchmark tournament
N_EPISODES = 5
results = []

print(f"🚀 Running {N_EPISODES}-episode Benchmark Tournament...")

for ep in range(N_EPISODES):
    env = make("kaggriculture", debug=False)
    env.run([agromaster_agent, starter_wheat_agent])
    
    p0_score = env.steps[-1][0]['observation']['farms'][0]['money']
    p1_score = env.steps[-1][1]['observation']['farms'][1]['money']
    winner = "AgroMaster-v1" if p0_score > p1_score else ("Starter Wheat" if p1_score > p0_score else "Tie")
    
    results.append({
        "Episode": ep + 1,
        "AgroMaster-v1 ($)": p0_score,
        "Starter Wheat ($)": p1_score,
        "Winner": winner
    })

df_results = pd.DataFrame(results)
print("\n🏆 Tournament Summary Results:")
print(df_results.to_string(index=False))

# Plot performance distribution
plt.figure(figsize=(10, 5))
df_melted = df_results.melt(id_vars=["Episode", "Winner"], value_vars=["AgroMaster-v1 ($)", "Starter Wheat ($)"], var_name="Agent", value_name="Final Coins ($)")
sns.barplot(data=df_melted, x="Episode", y="Final Coins ($)", hue="Agent")
plt.title("Tournament Comparison: AgroMaster-v1 vs. Starter Wheat Baseline", fontsize=14, fontweight='bold')
plt.xlabel("Episode Number", fontsize=12)
plt.ylabel("Final Coin Balance ($)", fontsize=12)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()


## 📦 Section 7: Agent Packaging & Kaggle Submission

To submit your agent to the Kaggle Leaderboard or run it on Kaggle Kernel Backend (KKB):

### 7.1 Agent Code Export (`main.py`)


In [ ]:
# Write standalone main.py agent file for submission
agent_code = """
import math

def agent(obs):
    player = obs["player"]
    day = obs["day"]
    me = obs["farms"][player]
    private = obs["private"]
    fx, fy = me["farmer"]
    tiles = me["tiles"]
    money = me["money"]
    market_obs = obs["market"]
    prices = market_obs["prices"]
    
    market_orders = []
    
    # 1. Market Operations
    target_crop = "WHEAT" if day < 8 else ("TOMATO" if money > 80 else "CARROT")
    target_seed_cost = 10 if target_crop == "WHEAT" else (50 if target_crop == "TOMATO" else 20)

    if private["seeds"].get(target_crop, 0) < 3 and money >= target_seed_cost * 3:
        market_orders.append(["BUY_SEED", target_crop, 3])
    if private["seeds"].get("WHEAT", 0) < 2 and money >= 20:
        market_orders.append(["BUY_SEED", "WHEAT", 2])

    for item, qty in private["shed"].items():
        if qty > 0 and (prices.get(item, 0) > 1 or day >= 27):
            market_orders.append(["SELL", item, qty])

    if day > 3 and me["hires_today"] == 0 and money > 500:
        market_orders.append(["HIRE", 1])

    # 2. Farmer Movement & Actions
    current_tile = tiles[fy][fx]
    if isinstance(current_tile, dict) and current_tile.get("kind") == "WEED":
        return {"farmer": ["DIG"], "hands": [], "market": market_orders}

    if isinstance(current_tile, dict) and current_tile.get("kind") == "PLANT":
        crop = current_tile["crop"]
        crop_age = day - current_tile["planted_day"]
        if crop in ["WHEAT", "CARROT", "MELON"] and crop_age >= 2:
            return {"farmer": ["HARVEST"], "hands": [], "market": market_orders}
        elif crop in ["TOMATO", "STRAWBERRY"] and crop_age >= 8:
            return {"farmer": ["HARVEST"], "hands": [], "market": market_orders}
        if not current_tile.get("watered_today", False):
            return {"farmer": ["WATER"], "hands": [], "market": market_orders}

    if current_tile is None:
        if private["seeds"].get(target_crop, 0) > 0:
            return {"farmer": ["PLANT", target_crop], "hands": [], "market": market_orders}
        elif private["seeds"].get("WHEAT", 0) > 0:
            return {"farmer": ["PLANT", "WHEAT"], "hands": [], "market": market_orders}

    # BFS Pathfinding
    queue = [(fx, fy, [])]
    visited = set([(fx, fy)])
    while queue:
        cx, cy, path = queue.pop(0)
        tile_state = tiles[cy][cx]
        if (cx, cy) != (fx, fy):
            if tile_state is None and (private["seeds"].get(target_crop, 0) > 0 or private["seeds"].get("WHEAT", 0) > 0):
                return {"farmer": [path[0]], "hands": [], "market": market_orders}
            if isinstance(tile_state, dict) and (tile_state.get("kind") == "WEED" or (tile_state.get("kind") == "PLANT" and not tile_state.get("watered_today", False))):
                return {"farmer": [path[0]], "hands": [], "market": market_orders}
        
        for nx, ny, direction in [(cx, cy-1, "NORTH"), (cx, cy+1, "SOUTH"), (cx+1, cy, "EAST"), (cx-1, cy, "WEST")]:
            if 0 <= nx < 10 and 0 <= ny < 10 and tiles[ny][nx] != "LOCKED" and (nx, ny) not in visited:
                visited.add((nx, ny))
                queue.append((nx, ny, path + [direction]))

    return {"farmer": ["PASS"], "hands": [], "market": market_orders}
"""

with open("main.py", "w") as f:
    f.write(agent_code.strip())

print("✅ main.py written successfully!")


In [ ]:
# Test running main.py with kaggle-environments
env = make("kaggriculture", debug=True)
env.run(["main.py", starter_wheat_agent])
print(f"✅ Local validation pass! Final Rewards: {[s['reward'] for s in env.steps[-1]]}")


## 🏁 Conclusion & Next Steps

We now have a working, highly competitive baseline for **Kaggriculture**.

### Key Takeaways & Improvement Directions:
1. **Dynamic Town Shop Exploitation**: Monitor `obs["town"]["unlocked_shops"]` and prioritize crops demanded by unlocked shops to benefit from price inflation.
2. **Livestock & Care Bonus Loops**: Implement Coops & Pastures, using Wheat to feed Geese/Cows/Sheep and stacking `CARE` bonuses for massive high-value yield spikes.
3. **Multi-Agent Coordination**: Utilize hired farm hands (`hands`) efficiently by assigning separate BFS paths to hands and farmer.
4. **Opponent Observation**: Track opponent farm layouts (`farms[1-player]`) to predict upcoming supply gluts and front-run sell orders.

Happy Farming! 🌾🚜🎉
